<a href="https://colab.research.google.com/github/RiverAlpha/DL/blob/main/2_1_Data_Manipulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 数据操作

In [1]:
import torch

In [2]:
# 生成一个长度为12，数据类型是float32的向量
x = torch.arange(12, dtype=torch.float32)
x

tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11.])

如何判断该使用什么精度？
选择哪种精度，本质上是在**“内存/计算速度”和“数值准确度”**之间做取舍。你可以根据以下场景来判断：

场景一：常规的模型训练与测试（默认选择）
建议使用：torch.float32

原因：这是 PyTorch 的默认设置，绝大多数经典的卷积神经网络（CNN）、基础的 Transformer 都不需要你去修改它。直接使用它最安全，不会遇到奇怪的 NaN（非数字）或数值下溢问题。

场景二：显存不够用，或者想大幅提升训练速度
建议使用：自动混合精度训练（AMP）结合 torch.float16 或 torch.bfloat16

原因：现在的英伟达显卡（如 RTX 30/40 系列，A100, H100）内部都有专门加速 16位 运算的 Tensor Core。

怎么选 16位：

如果你的显卡是 Ampere 架构及以上（例如 RTX 30 系列及以上、A100），强烈建议优先使用 torch.bfloat16。它不仅快，而且不容易出现梯度溢出。

如果是较老的显卡（如 V100, RTX 20 系列），硬件不支持 bfloat16，则使用 torch.float16 配合 torch.cuda.amp.GradScaler 来防止数值下溢。

场景三：科学计算或对数值极其敏感的任务
建议使用：torch.float64

原因：如果你在解偏微分方程、进行复杂的矩阵求逆、或是做物理/金融模拟，32位的舍入误差可能会在多次迭代后被无限放大。这时候必须牺牲速度，使用双精度。

场景四：模型部署与推理（Inference），希望极限压缩模型大小
建议使用：torch.int8 或 torch.float16 (模型量化)

原因：当模型训练完毕后，参数就固定了。此时可以将模型权重从 32 位浮点数强制转换为 8位整数（INT8）或 16位浮点数（FP16）。这可以让模型体积缩小 2 到 4 倍，推理速度翻倍，非常适合部署在手机端或算力有限的边缘设备上。

场景五：处理类别标签或索引
建议使用：torch.int64 (LongTensor)

原因：在使用 nn.CrossEntropyLoss 计算交叉熵损失，或者使用 tensor[index] 进行索引切片时，PyTorch 底层 API 严格要求输入类型为 long。

In [3]:
# 查看张量元素个数
x.numel()

12

In [4]:
# 查看张量的形状
x.shape

torch.Size([12])

## 张量形状改变

In [5]:
X = x.reshape(3,4)
X

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])

## 生成指定形状的0元素矩阵

In [6]:
torch.zeros((2,3,4))

tensor([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]])

In [7]:
torch.ones((2,3,4))

tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])

## randn -> rand norm

In [8]:
## 随机生成
torch.randn(3,4)

tensor([[-1.8007,  0.0407,  0.6858,  0.7851],
        [-0.5280, -1.7576,  0.1932,  0.4266],
        [ 1.1315, -0.1958, -0.7497, -0.5369]])

## 将指定的矩阵转化为张量类型

In [9]:
torch.tensor([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

tensor([[2, 1, 4, 3],
        [1, 2, 3, 4],
        [4, 3, 2, 1]])

## 索引和切片

In [10]:
X[-1],X[1:3]

(tensor([ 8.,  9., 10., 11.]),
 tensor([[ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]))

In [11]:
X[1,2] = 17
X

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5., 17.,  7.],
        [ 8.,  9., 10., 11.]])

In [12]:
X[:2,:1] = 12
X

tensor([[12.,  1.,  2.,  3.],
        [12.,  5., 17.,  7.],
        [ 8.,  9., 10., 11.]])

# 运算

In [13]:
torch.exp(x)

tensor([1.6275e+05, 2.7183e+00, 7.3891e+00, 2.0086e+01, 1.6275e+05, 1.4841e+02,
        2.4155e+07, 1.0966e+03, 2.9810e+03, 8.1031e+03, 2.2026e+04, 5.9874e+04])

In [14]:
x = torch.tensor([1.0, 2, 4, 8])
y = torch.tensor([2, 2, 2, 2])
# 元素会一一对应进行运算
x + y, x - y, x * y, x / y, x ** y

(tensor([ 3.,  4.,  6., 10.]),
 tensor([-1.,  0.,  2.,  6.]),
 tensor([ 2.,  4.,  8., 16.]),
 tensor([0.5000, 1.0000, 2.0000, 4.0000]),
 tensor([ 1.,  4., 16., 64.]))

## dim=0/1

In [15]:
X = torch.arange(12, dtype=torch.float32).reshape((3,4))
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
# dim=0代表纵向，dim=1代表横向拓展
torch.cat((X, Y), dim=0), torch.cat((X, Y), dim=1)

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.],
         [ 2.,  1.,  4.,  3.],
         [ 1.,  2.,  3.,  4.],
         [ 4.,  3.,  2.,  1.]]),
 tensor([[ 0.,  1.,  2.,  3.,  2.,  1.,  4.,  3.],
         [ 4.,  5.,  6.,  7.,  1.,  2.,  3.,  4.],
         [ 8.,  9., 10., 11.,  4.,  3.,  2.,  1.]]))

In [16]:
# 掩码运算
X == Y

tensor([[False,  True, False,  True],
        [False, False, False, False],
        [False, False, False, False]])

In [17]:
X.sum()

tensor(66.)

# 广播机制

In [18]:
a = torch.arange(3).reshape((3, 1))
b = torch.arange(2).reshape((1, 2))
a, b

(tensor([[0],
         [1],
         [2]]),
 tensor([[0, 1]]))

In [25]:
a+b

tensor([[[0, 2],
         [2, 4]],

        [[4, 6],
         [6, 8]]])

# 数据类型转化

核心机制（内存共享）：这种转换极其高效，因为当 X 位于 CPU 时，X、A 和 B 共享同一块物理内存。这意味着，如果你修改了 NumPy 数组 A 中的某个数值，张量 B 和原始的 X 也会同步发生改变，因为它们本质上是指向同一批数据的不同“外壳”。

关于转换的“直接方法”与“隐藏陷阱”
.numpy() 确实是转为 NumPy 的直接方法，但在真实的深度学习代码中，直接写 X.numpy() 往往会报错。你需要注意两个最常见的陷阱：

陷阱 1：张量在 GPU 上
NumPy 是基于 CPU 计算的库，无法读取显存中的数据。如果 X 在显卡上，直接转换会报错。
解决：必须先把它拉回内存，即 X.cpu().numpy()。

陷阱 2：张量绑定了计算图（带有梯度）
如果 X 是模型前向传播产生的结果（其 requires_grad=True），PyTorch 为了保护反向传播的安全，不允许它直接脱离 PyTorch 的控制转成 NumPy。
解决：必须先把它从计算图中“剥离”出来，即 X.detach().numpy()。

因此，在实际工程中，最稳妥、最常见的万能转换写法是：
A = X.detach().cpu().numpy()

In [19]:
A = X.numpy()
B = torch.from_numpy(A)
type(A),type(B)

(numpy.ndarray, torch.Tensor)

In [20]:
a = torch.tensor([3.5])
a, a.item, float(a), int(a)

(tensor([3.5000]), <function Tensor.item>, 3.5, 3)

#Ex

1. Run the code in this section. Change the conditional statement X == Y to X < Y or X > Y, and then see what kind of tensor you can get.

In [21]:
X <Y

tensor([[ True, False,  True, False],
        [False, False, False, False],
        [False, False, False, False]])

In [22]:
X>Y

tensor([[False, False, False, False],
        [ True,  True,  True,  True],
        [ True,  True,  True,  True]])

2. Replace the two tensors that operate by element in the broadcasting mechanism with other shapes, e.g., 3-dimensional tensors. Is the result the same as expected?

In [27]:
a = torch.arange(8).reshape((2, 2, 2))
b = torch.arange(2).reshape((1, 2,1))
a, b

(tensor([[[0, 1],
          [2, 3]],
 
         [[4, 5],
          [6, 7]]]),
 tensor([[[0],
          [1]]]))

In [28]:
a+b

tensor([[[0, 1],
         [3, 4]],

        [[4, 5],
         [7, 8]]])